# Stage 3 — Evaluate
**Twitter Sentiment Analysis MLOps · MAI201**

Notebook version of `src/evaluate.py` (DVC stage `evaluate`).

Scores `models/model.pkl` on the held-out test set. Metrics are written to
`metrics/scores.json` (DVC-tracked) **and** logged to the **same MLflow run**
that `02_train.ipynb` created (via `models/run_id.txt`).

**Inputs:** `models/model.pkl`, `data/processed/test.csv` · **Outputs:** `metrics/scores.json`, `metrics/confusion_matrix.json`, MLflow test metrics

In [ ]:
import json, os, pickle
import pandas as pd
import yaml
import mlflow
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

with open("params.yaml", encoding="utf-8") as f:
    ml_cfg = yaml.safe_load(f)["mlflow"]
mlflow.set_tracking_uri(ml_cfg["tracking_uri"])
mlflow.set_experiment(ml_cfg["experiment_name"])

with open("models/model.pkl", "rb") as f:
    pipeline = pickle.load(f)

test_df = pd.read_csv("data/processed/test.csv")
X, y = test_df["text"].astype(str), test_df["label"]
print(f"Test rows: {len(test_df):,}")

## Test metrics

In [ ]:
preds = pipeline.predict(X)

metrics = {
    "accuracy":  accuracy_score(y, preds),
    "precision": precision_score(y, preds),
    "recall":    recall_score(y, preds),
    "f1":        f1_score(y, preds),
}
clf = pipeline.named_steps["clf"]
if hasattr(clf, "predict_proba"):
    metrics["auc_roc"] = roc_auc_score(y, pipeline.predict_proba(X)[:, 1])
elif hasattr(clf, "decision_function"):
    metrics["auc_roc"] = roc_auc_score(y, pipeline.decision_function(X))

print(json.dumps(metrics, indent=2))
print()
print(classification_report(y, preds, target_names=["negative", "positive"]))

## Confusion matrix

In [ ]:
cm = confusion_matrix(y, preds)
ConfusionMatrixDisplay(cm, display_labels=["negative", "positive"]).plot(cmap="Blues")
plt.title("Confusion matrix — test set")
plt.show()

## Save metrics (DVC) + attach to the MLflow run from stage 2

In [ ]:
os.makedirs("metrics", exist_ok=True)
with open("metrics/scores.json", "w") as f:
    json.dump(metrics, f, indent=2)
with open("metrics/confusion_matrix.json", "w") as f:
    json.dump({"labels": ["negative", "positive"], "matrix": cm.tolist()}, f, indent=2)

with open("models/run_id.txt") as f:
    run_id = f.read().strip()
with mlflow.start_run(run_id=run_id):
    mlflow.log_metrics({f"test_{k}": v for k, v in metrics.items()})
    mlflow.log_artifact("metrics/confusion_matrix.json")

print(f"Logged test metrics to MLflow run {run_id}")
print("View runs:  mlflow ui --backend-store-uri sqlite:///mlflow.db")